In [1]:
import torch
import sklearn  
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
sys.path.append('/gpfs0/bgu-benshimo/users/wavishay/thesis/avishai_work/ASV_utils/')
from data_loading import load_data_all, load_data_male, load_data_female
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap 

import matplotlib.pyplot as plt
from matplotlib.widgets import Slider

from ipywidgets import Dropdown, FloatSlider, interact, Layout, SelectionSlider
from ipywidgets import interact, IntSlider


# Gender recognization ASVSpoof2019:

In [2]:
class class_time_embeddings_gender_classification:
    def __init__(self,path_to_time_embeddings,include_eval=False,include_dev=False):
        self.path_to_time_embeddings = path_to_time_embeddings
        embedded_groups_1_1,embedded_groups_1_2,embedded_groups_1_3, \
        chosen_labels_1_1_is_spoofed,chosen_labels_2_1_is_spoofed,chosen_labels_3_1_is_spoofed, \
        chosen_labels_numeric_1_1,chosen_labels_numeric_2_1,chosen_labels_numeric_3_1, \
        chosen_labels_1_1_attack_logical,chosen_labels_2_1_attack_logical,chosen_labels_3_1_attack_logical, \
        chosen_labels_1_1_name,chosen_labels_2_1_name,chosen_labels_3_1_name, \
        chosen_labels_1_1_speaker_id,chosen_labels_2_1_speaker_id,chosen_labels_3_1_speaker_id, \
        chosen_labels_1_1_sex,chosen_labels_2_1_sex,chosen_labels_3_1_sex = load_data_all(self.path_to_time_embeddings,include_eval =include_eval,include_dev = include_dev)
        self.embedded_groups_1_1 = embedded_groups_1_1
        self.embedded_groups_1_2 = embedded_groups_1_2
        self.embedded_groups_1_3 = embedded_groups_1_3
        self.chosen_labels_1_1_is_spoofed = chosen_labels_1_1_is_spoofed
        self.chosen_labels_2_1_is_spoofed = chosen_labels_2_1_is_spoofed
        self.chosen_labels_3_1_is_spoofed = chosen_labels_3_1_is_spoofed
        self.chosen_labels_numeric_1_1 = chosen_labels_numeric_1_1
        self.chosen_labels_numeric_2_1 = chosen_labels_numeric_2_1
        self.chosen_labels_numeric_3_1 = chosen_labels_numeric_3_1
        self.chosen_labels_1_1_attack_logical = chosen_labels_1_1_attack_logical
        self.chosen_labels_2_1_attack_logical = chosen_labels_2_1_attack_logical
        self.chosen_labels_3_1_attack_logical = chosen_labels_3_1_attack_logical
        self.chosen_labels_1_1_name = chosen_labels_1_1_name
        self.chosen_labels_2_1_name = chosen_labels_2_1_name
        self.chosen_labels_3_1_name = chosen_labels_3_1_name
        self.chosen_labels_1_1_speaker_id = chosen_labels_1_1_speaker_id
        self.chosen_labels_2_1_speaker_id = chosen_labels_2_1_speaker_id
        self.chosen_labels_3_1_speaker_id = chosen_labels_3_1_speaker_id
        self.chosen_labels_1_1_sex = chosen_labels_1_1_sex
        self.chosen_labels_2_1_sex = chosen_labels_2_1_sex
        self.chosen_labels_3_1_sex = chosen_labels_3_1_sex
        self.train_embedding_pca = None
        self.dev_embedding_pca = None
        self.train_embedding_tsne = None
        self.dev_embedding_tsne = None
        
        self.chosen_labels_1_1_attack_logical = pd.Series([x[0] for x in self.chosen_labels_1_1_attack_logical])
        
        self.chosen_labels_2_1_attack_logical = pd.Series([x[0] for x in self.chosen_labels_2_1_attack_logical])
        
        self.chosen_labels_3_1_attack_logical = pd.Series([x[0] for x in self.chosen_labels_3_1_attack_logical])
        
        self.chosen_labels_1_1_sex = pd.Series([x[0] for x in self.chosen_labels_1_1_sex])
        
        self.chosen_labels_2_1_sex = pd.Series([x[0] for x in self.chosen_labels_2_1_sex])
        
        self.chosen_labels_3_1_sex = pd.Series([x[0] for x in self.chosen_labels_3_1_sex])
                                               
 
        
        # Define unique labels including 'genuine'
        unique_labels = np.concatenate([self.chosen_labels_1_1_attack_logical.unique() , self.chosen_labels_2_1_attack_logical.unique()], axis=0)
        
        # Generate a colormap
        colors = plt.cm.viridis(np.linspace(0, 1, len(unique_labels)))
        
        # Map each unique label to a color
        self.label_to_color = {label: color for label, color in zip(unique_labels, colors)}
        
        
        self.chosen_labels_1_1_attack_logical_mapping = self.chosen_labels_1_1_attack_logical.map(self.label_to_color)
       
        self.chosen_labels_2_1_attack_logical_mapping = self.chosen_labels_2_1_attack_logical.map(self.label_to_color)
        
        self.frontsize = 16
        
          
    def create_UMAP(self,include_dev=False,include_eval=False):
        
        
        custom_params = {
            'n_components': 3,
            # 'n_neighbors': 15,
            # 'min_dist': 0.3,
            # 'metric': 'cosine',
            'n_jobs': -1,
            'random_state': 42
        }
        
        umap_model_train = umap.UMAP(**custom_params)
        umap_model_train.fit(self.embedded_groups_1_1)
        self.train_embedding_umap = umap_model_train.transform(self.embedded_groups_1_1 )
        
        if include_dev:
            umap_model_dev = umap.UMAP(**custom_params)
            umap_model_dev.fit(self.embedded_groups_1_2)
            self.dev_embedding_umap = umap_model_dev.transform(self.embedded_groups_1_2)
            
        if include_eval:
            umap_model_eval = umap.UMAP(**custom_params)
            umap_model_eval.fit(self.embedded_groups_1_3)
            self.eval_embedding_umap = umap_model_eval.transform(self.embedded_groups_1_3)
        
        return self.train_embedding_umap, self.dev_embedding_umap, self.eval_embedding_umap
    
           
    def create_pca(self,include_dev=False,include_eval=False):
        pca = PCA(n_components=3)
        pca.fit(self.embedded_groups_1_1)
        self.train_embedding_pca = pca.transform(self.embedded_groups_1_1 )
        
        if include_dev:
            pca = PCA(n_components=3)
            pca.fit(self.embedded_groups_1_2)
            self.dev_embedding_pca = pca.transform(self.embedded_groups_1_2)
            
        if include_eval:
            pca = PCA(n_components=3)
            pca.fit(self.embedded_groups_1_3)
            self.eval_embedding_pca = pca.transform(self.embedded_groups_1_3)
        
        print('PCA explained variance ratio:',pca.explained_variance_ratio_)
        return self.train_embedding_pca ,self.dev_embedding_pca,self.eval_embedding_pca
    
    
   

    def plotting_train(self, pca_or_t_sne='umap', plot_title='Train embeddings'):
        label_to_color = {'male': 'green', 'female': 'orange'}
        color_data = np.array([label_to_color[label] for label in self.chosen_labels_1_1_sex])

        if pca_or_t_sne == 'pca':
            coords = np.column_stack([
                self.train_embedding_pca[:, 0],
                -self.train_embedding_pca[:, 1],
                -self.train_embedding_pca[:, 2]
            ])
            labels = ['PCA 1', 'PCA 2', 'PCA 3']
        elif pca_or_t_sne == 't_sne':
            coords = np.column_stack([
                -self.train_embedding_tsne[:, 0],
                -self.train_embedding_tsne[:, 1],
                -self.train_embedding_tsne[:, 2]
            ])
            labels = ['t-SNE 1', 't-SNE 2', 't-SNE 3']
        elif pca_or_t_sne == 'umap':
            coords = self.train_embedding_umap
            labels = ['UMAP 1', 'UMAP 2', 'UMAP 3']
        else:
            print("Please choose between 'pca', 't_sne', or 'umap'")
            return

        def plot_func(azim=75, elev=30):
            fig = plt.figure(figsize=(10, 8))
            ax = fig.add_subplot(111, projection='3d')
            ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=color_data, alpha=0.5)
            ax.set_xlabel(labels[0])
            ax.set_ylabel(labels[1])
            ax.set_zlabel(labels[2])
            ax.set_title(plot_title, fontsize=20)
            ax.view_init(azim=azim, elev=elev)

            handles = [
                plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10)
                for label, color in label_to_color.items()
            ]
            ax.legend(handles, ['Male', 'Female'], title="Labels", fontsize=self.frontsize, title_fontsize='18', loc='lower right')
            plt.show()

        interact(plot_func,
                azim=IntSlider(min=0, max=360, step=5, value=75),
                elev=IntSlider(min=0, max=90, step=5, value=30))


        
    
    def plotting_dev(self, pca_or_t_sne='umap', plot_title='Dev embeddings'):
        label_to_color = {'male': 'purple', 'female': 'black'}
        color_data = np.array([label_to_color[label] for label in self.chosen_labels_2_1_sex])

        if pca_or_t_sne == 'pca':
            coords = np.column_stack([
                self.dev_embedding_pca[:, 0],
                -self.dev_embedding_pca[:, 1],
                -self.dev_embedding_pca[:, 2]
            ])
            labels = ['PCA 1', 'PCA 2', 'PCA 3']
        elif pca_or_t_sne == 't_sne':
            coords = np.column_stack([
                self.dev_embedding_tsne[:, 0],
                -self.dev_embedding_tsne[:, 1],
                -self.dev_embedding_tsne[:, 2]
            ])
            labels = ['t-SNE 1', 't-SNE 2', 't-SNE 3']
        elif pca_or_t_sne == 'umap':
            coords = self.dev_embedding_umap
            labels = ['UMAP 1', 'UMAP 2', 'UMAP 3']
        else:
            print("Please choose between 'pca', 't_sne', or 'umap'")
            return

        def plot_func(azim=75, elev=30):
            fig = plt.figure(figsize=(10, 8))
            ax = fig.add_subplot(111, projection='3d')
            ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=color_data, alpha=0.5)
            ax.set_xlabel(labels[0])
            ax.set_ylabel(labels[1])
            ax.set_zlabel(labels[2])
            ax.set_title(plot_title, fontsize=20)
            ax.view_init(azim=azim, elev=elev)

            handles = [
                plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10)
                for label, color in label_to_color.items()
            ]
            ax.legend(handles, ['Male', 'Female'], title="Labels", fontsize=self.frontsize, title_fontsize='18', loc='lower right')
            plt.show()

        interact(plot_func,
                azim=IntSlider(min=0, max=360, step=5, value=75),
                elev=IntSlider(min=0, max=90, step=5, value=30))


        
    def plotting_eval(self, pca_or_t_sne='umap', plot_title='Eval embeddings'):
        label_to_color = {'male': 'red', 'female': 'pink'}
        color_data = np.array([label_to_color[label] for label in self.chosen_labels_3_1_sex])

        if pca_or_t_sne == 'pca':
            coords = np.column_stack([
                self.eval_embedding_pca[:, 0],
                -self.eval_embedding_pca[:, 1],
                -self.eval_embedding_pca[:, 2]
            ])
            labels = ['PCA 1', 'PCA 2', 'PCA 3']
        elif pca_or_t_sne == 't_sne':
            coords = np.column_stack([
                self.eval_embedding_tsne[:, 0],
                self.eval_embedding_tsne[:, 1],
                -self.eval_embedding_tsne[:, 2]
            ])
            labels = ['t-SNE 1', 't-SNE 2', 't-SNE 3']
        elif pca_or_t_sne == 'umap':
            coords = self.eval_embedding_umap
            labels = ['UMAP 1', 'UMAP 2', 'UMAP 3']
        else:
            print("Please choose between 'pca', 't_sne', or 'umap'")
            return

        def plot_func(azim=75, elev=30):
            fig = plt.figure(figsize=(10, 8))
            ax = fig.add_subplot(111, projection='3d')
            ax.scatter(coords[:, 0], coords[:, 1], coords[:, 2], c=color_data, alpha=0.5)
            ax.set_xlabel(labels[0])
            ax.set_ylabel(labels[1])
            ax.set_zlabel(labels[2])
            ax.set_title(plot_title, fontsize=20)
            ax.view_init(azim=azim, elev=elev)

            handles = [
                plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10)
                for label, color in label_to_color.items()
            ]
            ax.legend(handles, ['Male', 'Female'], title="Labels", fontsize=self.frontsize, title_fontsize='18', loc='lower right')
            plt.show()

        interact(plot_func,
                azim=IntSlider(min=0, max=360, step=5, value=75),
                elev=IntSlider(min=0, max=90, step=5, value=30))

    
    def create_t_sne(self,include_dev=False,include_eval=False):
        tsne = TSNE(n_components=3,n_jobs=-1)
        self.train_embedding_tsne = tsne.fit_transform(self.embedded_groups_1_1)
        
        if include_dev:
            tsne = TSNE(n_components=3,n_jobs=-1)
            self.dev_embedding_tsne = tsne.fit_transform(self.embedded_groups_1_2)
        
        if include_eval:
            tsne = TSNE(n_components=3,n_jobs=-1)
            self.eval_embedding_tsne = tsne.fit_transform(self.embedded_groups_1_3)
        
        return self.train_embedding_tsne ,self.dev_embedding_tsne,self.eval_embedding_tsne

In [3]:
time_original = class_time_embeddings_gender_classification('/gpfs0/bgu-benshimo/users/wavishay/thesis/avishai_work/Data/male_vs_female_DB_models/16_bits/none/all/',include_dev=True,include_eval=True)

In [4]:
time_original.create_UMAP(include_dev=True,include_eval=True)
time_original.plotting_train(pca_or_t_sne='umap',plot_title='Training Set')

/gpfs0/bgu-benshimo/users/wavishay/env/pythn_new/lib/python3.9/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/gpfs0/bgu-benshimo/users/wavishay/env/pythn_new/lib/python3.9/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/gpfs0/bgu-benshimo/users/wavishay/env/pythn_new/lib/python3.9/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


interactive(children=(IntSlider(value=75, description='azim', max=360, step=5), IntSlider(value=30, descriptio…

In [5]:
time_original.plotting_dev(pca_or_t_sne='umap',plot_title='Development Set')


interactive(children=(IntSlider(value=75, description='azim', max=360, step=5), IntSlider(value=30, descriptio…

In [6]:
time_original.plotting_eval(pca_or_t_sne='umap',plot_title='Evaluation Set')

interactive(children=(IntSlider(value=75, description='azim', max=360, step=5), IntSlider(value=30, descriptio…

In [7]:
class CustomUMAPEmbeddings(class_time_embeddings_gender_classification):
    def create_UMAP(self, include_dev=True, include_eval=True):
        print("Using custom UMAP configuration...")

        # Custom parameters for UMAP (example: different n_neighbors, metric, min_dist)
        custom_params = {
            'n_components': 3,
            # 'n_neighbors': 15,
            # 'min_dist': 0.3,
            # 'metric': 'cosine',
            'n_jobs': -1,
            'random_state': 42
        }

        umap_model_train = umap.UMAP(**custom_params)
        umap_model_train.fit(self.embedded_groups_1_1)
        self.train_embedding_umap = umap_model_train.transform(self.embedded_groups_1_1)

        if include_dev:
            self.dev_embedding_umap = umap_model_train.transform(self.embedded_groups_1_2)

        if include_eval:
            self.eval_embedding_umap = umap_model_train.transform(self.embedded_groups_1_3)

        return self.train_embedding_umap, self.dev_embedding_umap, self.eval_embedding_umap


In [8]:
umap_on_other_sets = CustomUMAPEmbeddings(path_to_time_embeddings='/gpfs0/bgu-benshimo/users/wavishay/thesis/avishai_work/Data/male_vs_female_DB_models/16_bits/none/all/', include_dev=True, include_eval=True)

In [9]:
umap_on_other_sets.create_UMAP(include_dev=True,include_eval=True)
umap_on_other_sets.plotting_train(pca_or_t_sne='umap',plot_title='Training Set')


Using custom UMAP configuration...


/gpfs0/bgu-benshimo/users/wavishay/env/pythn_new/lib/python3.9/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


interactive(children=(IntSlider(value=75, description='azim', max=360, step=5), IntSlider(value=30, descriptio…

In [10]:
umap_on_other_sets.plotting_dev(pca_or_t_sne='umap',plot_title='Development Set')


interactive(children=(IntSlider(value=75, description='azim', max=360, step=5), IntSlider(value=30, descriptio…

In [11]:
umap_on_other_sets.plotting_eval(pca_or_t_sne='umap',plot_title='Evaluation Set')

interactive(children=(IntSlider(value=75, description='azim', max=360, step=5), IntSlider(value=30, descriptio…